<a href="https://colab.research.google.com/github/feixh/colab-notebooks/blob/main/tiny_shapespeare.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Implement and train a dense (but tiny) language model


TODO: Practice explaining the techniques using precise and concise language


## Install dependencies and set up environment

In [ ]:
!pip install loguru einsum jaxtyping ipytest tiktoken


In [ ]:
import json
import pickle
import tqdm
import random
from pathlib import Path
from loguru import logger
import ipytest
ipytest.autoconfig(addopts=['-W ignore::pytest.PytestAssertRewriteWarning'])

from google.colab import drive
drive.mount("/content/drive")

project_dir = Path("/content/drive/My Drive/Colab Notebooks/ml_coding_prep")
workspace = project_dir / "workspace_tiny_shapespeare"
if workspace.exists():
  logger.warning(f"Workspace already exists @ {workspace} -- Make sure you actually want to use the same workspace!!!")
else:
  workspace.mkdir(parents=True, exist_ok=False)
  logger.info(f"workspace created @ {workspace}")


## Data preparation

Download the tiny Shapespeare dataset.

In [ ]:
raw_data_path = workspace / "tinyshakespeare.txt"

if not raw_data_path.exists():
  import requests

  # URL of the tiny shakespeare dataset
  url = 'https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt'

  # Download the dataset
  response = requests.get(url)
  response.raise_for_status() # Raise an exception for HTTP errors

  # Save the dataset to a file (optional, but good for persistence)
  with open(raw_data_path, 'w', encoding='utf-8') as f:
      f.write(response.text)

  logger.info(f'tiny shakespeare dataset downloaded and saved to {raw_data_path}')

else:
  logger.info(f"tiny shakespeare dataset already exists @ {raw_data_path}")


# Load the dataset content into a string
with open(raw_data_path, 'r', encoding='utf-8') as f:
    text = f.read()

# Display the first 100 characters to verify
print(text[:100])
print(f"\nTotal length of the dataset: {len(text)} characters")


In [ ]:
from typing import Dict

TOKEN_UNKNOWN = "<|Unknown|>"

def create_character_vocabulary(text: str, save_vocab_path: Path):
  """ Create a character-level vocabulary from a body of text.

  text: The body of text to create the vocabulary from.

  """
  # Create a character tokenizer for tiny Shakespeare
  char_set = set()
  for char in tqdm.tqdm(text):
    char_set.add(char)

  print(f"\n{char_set=}")

  sorted_chars = sorted(list(char_set))
  print(f"{sorted_chars=}")
  vocab = {}
  for idx, char in enumerate(sorted_chars):
    vocab[char] = idx

  # Add the special tokens
  # TODO: how about end of sequence (EOS)?
  special_tokens = [TOKEN_UNKNOWN]
  for s_token in special_tokens:
    idx += 1
    vocab[s_token] = idx

  # Save the vocabulary for later use.
  with open(save_vocab_path, "w") as f:
    json.dump(vocab, f, indent=2)
    logger.info(f"Vocabulary saved to {save_vocab_path}")

  return vocab


def load_character_vocabulary(vocab_path: Path) -> Dict[str, int]:
  with open(vocab_path, "r") as f:
    vocab = json.load(f)
  return vocab


def create_idx_to_char_vocab(char_to_idx_vocab: Dict[str, int]) -> Dict[int, str]:
  return {idx: char for char, idx in char_to_idx_vocab.items()}


def character_tokenize(seq: str, vocab: Dict[str, int]):
  """
  Map each character to their ASIIC value.
  Use the following for special characters?
  """
  token_seq = []
  for char in seq:
    idx = vocab.get(char, vocab[TOKEN_UNKNOWN])
    token_seq.append(idx)

  return token_seq


def character_detokenize(token_seq: list[int], idx_to_char_vocab: Dict[int, str]) -> str:
  decoded = []
  for idx in token_seq:
    char = idx_to_char_vocab[idx]
    if char == TOKEN_UNKNOWN:
      char = "\u25A0" # Solid square
    decoded.append(char)

  return ''.join(decoded)

def test_character_tokenization(seq: str):
  vocab_path = workspace / "test_vocab.json"
  _vocab = create_character_vocabulary(text, vocab_path)

  char_to_idx_vocab = load_character_vocabulary(vocab_path)

  assert _vocab == char_to_idx_vocab

  idx_to_char_vocab = create_idx_to_char_vocab(char_to_idx_vocab)

  for char in char_to_idx_vocab:
    idx = char_to_idx_vocab[char]
    assert char == idx_to_char_vocab[idx]


  token_seq = character_tokenize(seq, vocab=char_to_idx_vocab)
  print("========================================")
  print("|| tests started")
  print("========================================")
  print(f"Input sequence: {seq}")
  print(f"Tokenized sequence: {token_seq}")
  decoded = character_detokenize(token_seq, idx_to_char_vocab=idx_to_char_vocab)
  print(f"Detokenized sequence: {decoded}")

test_character_tokenization("Howdy?}}{{}}")

In [ ]:
class CharacterTokenizer:
  def __init__(self, vocab_path: Path):
    self.vocab = load_character_vocabulary(vocab_path)
    self.idx_to_char_vocab = create_idx_to_char_vocab(self.vocab)
    self.vocab_size = len(self.vocab)

  def tokenize(self, seq: str) -> list[int]:
    return character_tokenize(seq, self.vocab)

  def detokenize(self, token_seq: list[int]) -> str:
    return character_detokenize(token_seq, self.idx_to_char_vocab)

In [ ]:
%%ipytest

def test_character_tokenizer():
  tokenizer = CharacterTokenizer(vocab_path=workspace / "vocab.json")
  seq = "Hello, world!"
  token_seq = tokenizer.tokenize(seq)
  _seq = tokenizer.detokenize(token_seq)
  assert seq == _seq

In [ ]:
def sample_fixed_length_text(text: str, seq_len: int, rng: random.Random):
  """ Given a body of text, randomly sample a sequence of characters of a given length.

  text: The body of text to sample from.
  seq_len: The length of the sequence to sample.
  rng: A random number generator that has been seeded.
  """
  text_len = len(text)
  start_idx = rng.randint(0, text_len - seq_len)
  seq = text[start_idx:start_idx + seq_len]
  return seq


def test_sample_fixed_length_text(text: str):
  rng = random.Random()
  rng_seed = 42

  for seq_len in [16, 32, 64, 128]:
    print(f"========================================")
    print(f"|| {seq_len=}")
    print(f"========================================")
    rng.seed(rng_seed)
    seq1 = sample_fixed_length_text(text, seq_len, rng)
    print(f"{seq1=}")

    # sample again with the same seed
    rng.seed(rng_seed)
    seq2 = sample_fixed_length_text(text, seq_len, rng)
    print(f"{seq2=}")

    assert seq1 == seq2, f"Sequences are not equal: {seq1} != {seq2}"


    # sample again, this sequence should be different
    seq3 = sample_fixed_length_text(text, seq_len, rng)
    print(f"{seq3=}")
    assert seq3 != seq1, f"Sequences should NOT be equal! This is probably a coincidence? {seq1=}, {seq3=}"

  logger.info("All tests passed!")

test_sample_fixed_length_text(text)



In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class TinyTextDataset(Dataset):
    def __init__(self,
                 text: str,
                 seq_len: int,
                 num_epochs: int = 1,
                 rng_seed: int | None = 42,
                 rng_state: tuple | None = None):
      self.text = text
      self.seq_len = seq_len
      self.num_epochs = num_epochs
      logger.info(f"{num_epochs=}")

      self.text_len = len(text)

      # Initialize the random number generator.
      self.rng = random.Random()
      assert not (rng_seed is None and rng_state is None), "Either rng_seed or rng_state must be provided"
      if rng_state is None:
        self.rng.seed(rng_seed)
        logger.info(f"RNG seed set to {rng_seed=}")
      else:
        self.rng.setstate(rng_state)
        logger.info(f"Restoed RNG state from {rng_state=}")

    def __len__(self):
      return self.text_len * self.num_epochs

    def __getitem__(self, idx):
      # In a truly stateful scenario, 'idx' might be ignored or used differently.
      # Here, we'll use the internal RNG to sample a sequence, making it stateful.
      sample = sample_fixed_length_text(self.text, self.seq_len, self.rng)
      return sample

    def serialize_state(self):
      rng_state = self.rng.getstate()
      state = {
          "text": self.text,  # Serializing the whole text dataset is only valid for very small dataset.
          "seq_len": self.seq_len,
          "num_epochs": self.num_epochs,
          "rng_state": rng_state
      }
      return state

    def serialize_state_to_pickle(self, pkl_path: str):
      state = self.serialize_state()
      pickle.dump(state, open(pkl_path, "wb"))
      logger.info(f"Serialized state to {pkl_path}")

    @classmethod
    def create_from_pickle(cls, pkl_path: str) -> "TinyTextDataset":
      """ Create the dataset from a json file that has the serialized dataset state.
      """
      state = pickle.load(open(pkl_path, "rb"))
      return cls.create_from_state(state)


    @classmethod
    def create_from_state(cls, state: Dict) -> "TinyTextDataset":
      return cls(
          text=state["text"],
          seq_len=state["seq_len"],
          num_epochs=state["num_epochs"],
          rng_state=state["rng_state"]
      )


def test_tiny_text_dataset(text):
  dataset1 = TinyTextDataset(text, seq_len=32, num_epochs=5, rng_seed=42, rng_state=None)
  sample1 = dataset1[0]

  dataset2 = TinyTextDataset(text, seq_len=32, num_epochs=5, rng_seed=1, rng_state=None)
  sample2 = dataset2[0]

  assert sample1 != sample2, f"Samples should not be equal: {sample1=}, {sample2=}"

  # serialize the state of dataset1
  serialized_dataset_state_path =  "test_dataset_serialization.pkl"
  dataset1.serialize_state_to_pickle(serialized_dataset_state_path)

  # create a new dataset from dataset1's state
  dataset1_restored = TinyTextDataset.create_from_pickle(serialized_dataset_state_path)

  for i in range(1, 10):
    # Using the same index to access dataset1 and dataset1_restored should result in the
    # same sample.
    sample1 = dataset1[i]
    sample1_restored = dataset1_restored[i]
    assert sample1 == sample1_restored, f"Samples should be equal: {sample1=}, {sample1_restored=}"

  logger.info("All tests passed.")


def test_tiny_text_dataset_with_tokenization(text):
  vocab_path = workspace / "vocab.json"
  vocab = load_character_vocabulary(vocab_path=vocab_path)
  idx_to_char_vocab = create_idx_to_char_vocab(vocab)

  dataset = TinyTextDataset(text, seq_len=32, rng_seed=42, rng_state=None)
  for idx in range(10):
    seq = dataset[idx]
    token_seq = character_tokenize(seq, vocab)
    decoded = character_detokenize(token_seq,idx_to_char_vocab=idx_to_char_vocab)
    # print("========================================")
    # print(f"|| test case {idx:02d}")
    # print("========================================")
    # print(f"sequence: {seq}")
    # print(f"tokens: {token_seq}")
    # print(f"decoed sequence: {decoded}")
    assert seq == decoded, f"Sequences should be equal: {seq=}, {decoded=}"

  logger.info("All tests passed.")


test_tiny_text_dataset(text)
test_tiny_text_dataset_with_tokenization(text)


In [ ]:
import torch
from torch.utils.data import DataLoader


def create_collate_fn():
  tokenizer = CharacterTokenizer(vocab_path=workspace / "vocab.json")

  def collate_fn(batch):
    token_ids = []
    position_idx = []
    for seq in batch:
      token_seq = tokenizer.tokenize(seq)
      token_ids.append(torch.tensor(token_seq, dtype=torch.long))
      position_idx.append(torch.arange(0, len(token_seq), dtype=torch.long))

    # shape: [B, S] where B is the batch size, and S is the sequence length
    token_ids = torch.stack(token_ids, dim=0)
    position_idx = torch.stack(position_idx, dim=0)

    out_batch = {
        "text": batch,
        "token_ids": token_ids,
        "position_idx": position_idx
    }
    return out_batch

  return collate_fn, tokenizer

def test_loading_tiny_text_dataset():
  # Assuming `text` and `workspace` are already defined from previous cells
  # And TinyTextDataset, character_tokenize, and load_character_vocabulary are defined.

  # Parameters for the dataset and dataloader
  seq_len = 64
  batch_size = 32

  # Create the dataset instance
  dataset = TinyTextDataset(text, seq_len=seq_len, num_epochs=10, rng_seed=42)

  # Create the DataLoader
  collate_fn, _ = create_collate_fn()
  dataloader = DataLoader(
      dataset,
      batch_size=batch_size,
      shuffle=False, # shuffle=False for reproducible demonstration
      collate_fn=collate_fn
  )

  logger.info(f"Created DataLoader with batch_size={batch_size} and seq_len={seq_len}")
  logger.info(f"Number of batches in DataLoader: {len(dataloader)}")

  for b_idx, batch in enumerate(dataloader):
    # print(batch)
    # breakpoint()
    assert "text" in batch
    assert "token_ids" in batch
    assert "position_idx" in batch
    assert batch["token_ids"].shape == (batch_size, seq_len)
    assert batch["position_idx"].shape == (batch_size, seq_len)

    if b_idx > 10:
      logger.info("Tested the first 10 batches, seems working fine")
      break


test_loading_tiny_text_dataset()




## BPE tokenizer

Using one of those off-the-shelf Byte-Pair Encoding (BPE) tokenizers.



In [ ]:
import tiktoken

# Get the tokenizer used by GPT-2 (a common BPE tokenizer)
enc = tiktoken.get_encoding("gpt2")

sample_text = "Before we proceed any further, hear me speak."
tokens = enc.encode(sample_text)
decoded = enc.decode(tokens)

print(f"Vocabulary size: {enc.n_vocab}")
print(f"Original: {sample_text}")
print(f"Tokens: {tokens}")
print(f"Decoded: {decoded}")


## Model definition


Import dependencies for model definition

In [ ]:
from torch import nn, Tensor
from jaxtyping import Float, Int
from einops import einsum, rearrange

### Layer definitions

Define and test the following layers:
1. Attention
2. Multi-Head Attention
3. Mlp (i.e., Feed Forward Network)
4. MoE (simple implementation, also outputs load balancing related terms)
5. Transformer layer that consists of Multi-Head Attention and Mlp/MoE

In [ ]:
from typing import Tuple
from torch import nn, Tensor
from jaxtyping import Float, Int
from einops import einsum, rearrange
from dataclasses import dataclass

class Attention(nn.Module):
  def __init__(self,
               dim: int,
               max_seq_len: int = 128,
               use_causal_mask: bool = True):
    super().__init__()
    self.dim = dim
    self.max_seq_len = max_seq_len
    self.use_causal_mask = use_causal_mask
    if use_causal_mask:
      causal_mask = torch.triu(
          torch.ones((max_seq_len, max_seq_len), dtype=torch.bool),
          diagonal=1)
      self.register_buffer("causal_mask", causal_mask)

    self.qkv = nn.Linear(dim, 3 * dim, bias=False)
    # (Optionally) we can dropout some attention
    self.proj = nn.Linear(dim, dim, bias=False)


  def forward(self, x: Float[Tensor, "B S D"]) -> Float[Tensor, "B S D"]:
    """
    x: shape [B, S, D]
    """
    seq_len = x.shape[1]
    assert seq_len <= self.max_seq_len, f"Sequence length {seq_len} is greater than max sequence length {self.max_seq_len}"

    qkv = self.qkv(x) # [B, S, 3*D]
    q, k, v = qkv.chunk(3, dim=-1)  # each has shape [B, S, D]
    dot_product = einsum(q, k, "b s_q d, b s_k d -> b s_q s_k") # [B, Sq, Sk] where Sq == Sk
    scaled_dot_product = dot_product / (self.dim ** 0.5)  # normalize by hidden dim
    if self.use_causal_mask:
      scaled_dot_product = scaled_dot_product.masked_fill(
          self.causal_mask[:seq_len, :seq_len],
          -torch.inf
      )

    attention = scaled_dot_product.softmax(dim=-1)
    out = einsum(attention, v, "b s_q s_k, b s_k d -> b s_q d")
    out = self.proj(out)  # final projection

    return out

class MultiHeadAttention(nn.Module):
  def __init__(self,
               dim: int,
               heads: int,
               max_seq_len: int = 128,
               dropout: float = 0.0,
               use_causal_mask: bool = True,
               use_pytorch_sdpa: bool = False):
    super().__init__()
    assert dim % heads == 0, "dim must be divisible by heads"
    self.dim = dim
    self.heads = heads
    self.head_dim = dim // heads

    self.max_seq_len = max_seq_len
    self.use_causal_mask = use_causal_mask
    self.use_pytorch_sdpa = use_pytorch_sdpa
    if use_causal_mask and not use_pytorch_sdpa:
      causal_mask = torch.triu(
          torch.ones((max_seq_len, max_seq_len), dtype=torch.bool),
          diagonal=1)
      self.register_buffer("causal_mask", causal_mask)

    if use_pytorch_sdpa:
      logger.info("Using Pytorch's built-in SDPA -- Scaled Dot Product Attention")

    self.qkv = nn.Linear(dim, 3 * dim, bias=False)
    self.proj = nn.Linear(dim, dim, bias=False)

    self.dropout = dropout
    if dropout > 0:
      self.attn_dropout = nn.Dropout(dropout)
      self.proj_dropout = nn.Dropout(dropout)
    else:
      self.attn_dropout = nn.Identity()
      self.proj_dropout = nn.Identity()

  def apply_init_trick(self, num_layers: int):
    """
    Apply additional initialization trick to proj.
    The trick: For a linear layer closing out a residual connection, use
    N(0, sigma) where sigma = 0.02 / sqrt(2 * num_layers) to initialize.
    """
    nn.init.normal_(self.proj.weight, mean=0.0, std=0.02 / math.sqrt(2 * num_layers))


  def forward(self, x: Float[Tensor, "B S D"]) -> Float[Tensor, "B S D"]:
    seq_len = x.shape[1]
    assert seq_len <= self.max_seq_len, f"Sequence length {seq_len} is greater than max sequence length {self.max_seq_len}"

    qkv = self.qkv(x)
    q, k, v = qkv.chunk(3, dim=-1)  # B S D
    # split along head dimension
    # Corrected pattern: (h d) to decompose 'dim' into 'heads' and 'head_dim'
    q, k, v = [rearrange(each,
                         "b s (h d) -> b h s d", # Changed 'hd' to '(h d)'
                         h=self.heads,
                         d=self.head_dim) # Pass explicit sizes for h and d
              for each in [q, k, v]]

    if not self.use_pytorch_sdpa:
      # my own scaled dot product attention implementation
      dot_product = einsum(q, k, "b h s_q d, b h s_k d -> b h s_q s_k")
      scaled_dot_product = dot_product / (self.head_dim ** 0.5)
      if self.use_causal_mask:
        scaled_dot_product = scaled_dot_product.masked_fill(
            self.causal_mask[:seq_len, :seq_len],
            -torch.inf
        )
      attention = scaled_dot_product.softmax(dim=-1)
      attention = self.attn_dropout(attention)
      out = einsum(attention, v, "b h s_q s_k, b h s_k d -> b h s_q d")
    else:
      # use pytorch's built-in implementation of SDPA
      out = nn.functional.scaled_dot_product_attention(
          q, k, v,
          dropout_p=self.dropout if self.training else 0.0,
          is_causal=self.use_causal_mask) # [B H S D]
    out = rearrange(out, "b h s d -> b s (h d)", h=self.heads, d=self.head_dim) # Changed 'hd' to '(h d)'
    out = self.proj(out)
    out = self.proj_dropout(out)
    return out


class Mlp(nn.Module):
  def __init__(self, dim: int, dropout: float = 0.0):
    super().__init__()
    self.linear1 = nn.Linear(dim, 4 * dim)
    self.linear2 = nn.Linear(4 * dim, dim)
    if dropout > 0:
      self.dropout = nn.Dropout(dropout)
    else:
      self.dropout = nn.Identity()

  def apply_init_trick(self, num_layers: int):
    """
    Apply additional initialization trick to linear2.
    The trick: For a linear layer closing out a residual connection, use
    N(0, sigma) where sigma = 0.02 / sqrt(2 * num_layers) to initialize.
    """
    nn.init.normal_(self.linear2.weight, mean=0.0, std=0.02 / math.sqrt(2 * num_layers))

  def forward(self, x: Float[Tensor, "B S D"]) -> Float[Tensor, "B S D"]:
    return self.dropout(self.linear2(nn.functional.gelu(self.dropout(self.linear1(x)))))


@dataclass
class LoadBalancingTerms:
  P: Float[Tensor, "num_experts"]
  """
  P[i] is the probability that expert i is active.
  P[i] = 1 / num_tokens * sum_{j loops over all tokens} (weight of expert i in token j)
  if expert i is not chosen among the top-k for token j, its weight is zero
  """

  f: Float[Tensor, "num_experts"]
  """
  f[i] is the faction of tokens passed to expert i.
  f[i] = 1 / num_experts * number_of_tokens_in_a_batch__passed_to_expert_i
  """

class MoELayer(nn.Module):
  def __init__(self, dim: int, num_experts: int, num_active_experts: int, dropout: float):
    super().__init__()
    self.dim = dim
    self.num_experts = num_experts
    self.k = num_active_experts

    logger.info(f"Using MoE layer with {num_experts=} and {num_active_experts=}")

    self.router = nn.Linear(dim, num_experts)
    # In the current implementation, dropout is disabled in each expert.
    # Instead, one dropout is applied after the expert outputs are aggregated.
    self.experts = nn.ModuleList([Mlp(dim, dropout=0.0) for _ in range(num_experts)])
    self.dropout = nn.Dropout(dropout)

  def apply_init_trick(self, num_layers: int):
    for expert in self.experts:
      expert.apply_init_trick(num_layers)

  def forward(self, x: Float[Tensor, "B S D"]) -> \
    Tuple[Float[Tensor, "B S D"], LoadBalancingTerms]:

    bs, sl = x.shape[:2]  # batch size, and sequence length
    num_tokens = bs * sl

    dtype = x.dtype
    device = x.device

    flatten_x = rearrange(x, "b s d -> (b s) d")
    flatten_logits = self.router(flatten_x) # [(B S), N] where N is #experts

    values, indices = torch.topk(flatten_logits, k=self.k, dim=-1)
    weights = torch.nn.functional.softmax(values, dim=-1)

    out = torch.zeros_like(flatten_x)

    P = torch.zeros((self.num_experts, ), dtype=dtype, device=device)
    f = torch.zeros_like(P)

    for idx, expert in enumerate(self.experts):
      # identify which tokens need to be passed to this expert
      mask = (indices == idx) # shape: [(B S), K]

      # compute: the fraction of tokens in the batch that expert idx has been chosen
      # as one of the top-k experts
      f[idx] = mask.sum() / num_tokens

      if mask.any():
        token_idx, expert_idx = torch.where(mask)
        _expert_out = expert(flatten_x[token_idx])  # [N, D]
        _expert_weight = weights[token_idx, expert_idx] # [N, ]
        out.index_add_(0, token_idx, _expert_weight.unsqueeze(-1) * _expert_out)

        # compute: each expert has a certain probability being assigned to the token
        # P[idx] is the average probability of expert idx being assigned tokens in the batch
        P[idx] = _expert_weight.sum() / num_tokens

    self.dropout(out)

    # shape back
    out = rearrange(out, "(b s) d -> b s d", b=bs)
    return out, LoadBalancingTerms(P=P, f=f)


class TransformerLayer(nn.Module):
  def __init__(self,
               dim: int,
               heads: int,
               max_seq_len: int = 128,
               use_causal_mask: bool = True,
               dropout: float = 0.0,
               num_experts: int = -1,
               num_active_experts: int = -1):
    super().__init__()

    if num_experts > 0:
      assert 0 < num_active_experts < num_experts, f"{num_experts=} but {num_active_experts=} -- the latter should be positive and less than the former"
      logger.info(f"{num_experts=}, {num_active_experts=}")

    self.dim = dim
    self.heads = heads

    self.norm1 = nn.RMSNorm(dim)
    self.attn = MultiHeadAttention(dim=dim,
                                   heads=heads,
                                   max_seq_len=max_seq_len,
                                   use_causal_mask=use_causal_mask,
                                   dropout=dropout,
                                   use_pytorch_sdpa=True
                                   )
    self.norm2 = nn.RMSNorm(dim)
    if num_experts < 0:
      self.mlp = Mlp(dim=dim, dropout=dropout)
    else:
      self.mlp = MoELayer(dim=dim, num_experts=num_experts, num_active_experts=num_active_experts, dropout=dropout)

  def forward(self, x: Float[Tensor, "B S D"]) -> Tuple[Float[Tensor, "B S D"], LoadBalancingTerms | None]:
    # x: [B, S, D]

    x = x + self.attn(self.norm1(x))
    mlp_out = self.mlp(self.norm2(x))
    if isinstance(mlp_out, tuple):
      mlp_out, load = mlp_out # unpack
    else:
      load = None

    x = x + mlp_out

    return x, load


### Tests for the layer definitions

Test the layers defind above.



In [ ]:
%%ipytest


def test_attention():
  dim = 8
  batch_size, seq_len = 8, 32
  attn = Attention(dim=dim)
  seq = torch.randn((batch_size, seq_len, dim))
  out = attn(seq)
  assert out.shape == seq.shape

  all_ones_seq = torch.ones_like(seq)
  out = attn(all_ones_seq)  # for all ones sequence, each token within the sequence should be identical
  diff = out[:, 1:, :] - out[:, :-1, :]
  # breakpoint()
  assert torch.allclose(diff, torch.zeros_like(diff), atol=1e-6)
  # print(out[0, :, 1])


def test_multi_head_attention():
  dim = 32
  heads = 8
  batch_size, seq_len = 8, 32
  attn = MultiHeadAttention(dim=dim, heads=heads)
  x = torch.randn((batch_size, seq_len, dim))
  out = attn(x)
  assert out.shape == x.shape

  all_ones_seq = torch.ones_like(x)
  out = attn(all_ones_seq)  # for all ones sequence, each token within the sequence should be identical
  diff = out[:, 1:, :] - out[:, :-1, :]
  # breakpoint()
  assert torch.allclose(diff, torch.zeros_like(diff), atol=1e-6)


def test_mlp():
  batch_size, seq_len, dim = 8, 16, 32
  mlp = Mlp(dim=dim)

  x = torch.randn((batch_size, seq_len, dim))
  out = mlp(x)
  assert x.shape == out.shape


def test_routing_logic():
  batch_size, seq_len, dim = 8, 32, 16
  num_experts, num_active_experts = 8, 4

  router = torch.nn.Linear(dim, num_experts)
  experts = [torch.nn.Linear(dim, dim) for _ in range(num_experts)]

  x = torch.randn((batch_size, seq_len, dim))  # [B, S, D]
  device = x.device

  flatten_x = rearrange(x, "b s d -> (b s) d")
  num_tokens = flatten_x.shape[0]

  values, indices = torch.topk(flatten_x, k=num_active_experts, dim=-1) # (b s) k
  weights = torch.softmax(values, dim=-1)

  out = torch.zeros_like(flatten_x)
  P = torch.zeros((num_experts, ), device=device)
  f = torch.zeros((num_experts, ), device=device)

  for idx, expert in enumerate(experts):
    # identify which tokens need to be passed to this expert
    mask = (indices == idx) # [(B S), K]
    f[idx] = mask.sum() / num_tokens

    if mask.any():
      token_idx, expert_idx = torch.where(mask)
      # logger.info(f"{token_idx.shape=}, {expert_idx.shape=}")
      # logger.info(f"{expert_idx=}")
      _expert_out = expert(flatten_x[token_idx])
      _expert_weight = weights[token_idx, expert_idx]
      P[idx] = _expert_weight.sum() / num_tokens

      assert torch.all(weights[mask] == _expert_weight)
      logger.info(f"{_expert_out.shape=}, {_expert_weight.shape=}")
      _out = _expert_weight.unsqueeze(-1) * _expert_out
      out.index_add_(0, token_idx, _out)

  out = rearrange(out, "(b s) d -> b s d", b=batch_size)
  assert out.shape == x.shape

  logger.info(f"{f=}")
  logger.info(f"{P=}")


def test_moe_layer():
  batch_size, seq_len, dim = 8, 32, 16
  num_experts, num_active_experts = 8, 4
  layer = MoELayer(dim=dim, num_experts=num_experts, num_active_experts=num_active_experts, dropout=0.1)
  x = torch.randn((batch_size, seq_len, dim))
  out, load = layer(x)

  assert out.shape == x.shape
  assert load is not None
  assert load.P.shape == (num_experts, )
  assert load.f.shape == (num_experts, )



def test_transformer_layer():
  batch_size, seq_len, dim = 8, 16, 32
  heads = 4
  layer = TransformerLayer(dim=dim, heads=heads)
  x = torch.randn((batch_size, seq_len, dim))
  out, load = layer(x)
  assert out.shape == x.shape
  assert load is None


def test_transformer_layer_with_moe():
  batch_size, seq_len, dim = 8, 16, 32
  heads = 4
  layer = TransformerLayer(dim=dim, heads=heads, num_experts=4, num_active_experts=2)
  x = torch.randn((batch_size, seq_len, dim))
  out, load = layer(x)
  assert out.shape == x.shape
  assert load is not None


### Position embedding [TODO: try to explain each to people]

Implement:
1. Learnable position embedding
2. Absolute position embedding
3. RoPE

In [ ]:
from torch._prims_common import DimsSequenceType
from enum import Enum
import math

class PositionEmbeddingType(Enum):
  LEARNABLE = 0
  ABSOLUTE = 1
  ROPE = 2

class LearnablePositionEmbedding(nn.Module):
  def __init__(self, dim: int, max_seq_len: int):
    super().__init__()
    self.dim = dim
    self.max_seq_len = max_seq_len

    self.pos_emb = nn.Embedding(max_seq_len, dim)

  def forward(self, x: Float[Tensor, "B S D"], idx: Int[Tensor, "B S"]) -> Float[Tensor, "B S D"]:
    out = x + self.pos_emb(idx)
    return out


def create_sinusoidal_position_embedding(
    dim: int,
    max_seq_len: int,
    base: int = 10_000) -> Float[Tensor, "S D"]:
  assert dim % 2 == 0
  half_dim = dim // 2
  pos = torch.arange(0, max_seq_len).unsqueeze(dim=-1) # [S, 1]
  freq = torch.exp(-math.log(base) * 2 * torch.arange(0, half_dim) / dim).unsqueeze(dim=0) # [1, D//2]
  theta = pos * freq # [S, D//2]
  sin_theta = torch.sin(theta)  # [S, D//2]
  cos_theta = torch.cos(theta)  # [S, D//2]
  emb = torch.empty((max_seq_len, dim)) # [S, D]
  emb[:, ::2] = sin_theta
  emb[:, 1::2] = cos_theta
  return emb

class AbsolutePositionEmbedding(nn.Module):
  def __init__(self, dim: int, max_seq_len: int, base: int = 10_000):
    super().__init__()
    self.dim = dim
    self.max_seq_len = max_seq_len
    self.base = base

    emb = create_sinusoidal_position_embedding(
        dim, max_seq_len, base
    )
    self.register_buffer("pos_emb", emb)

  def forward(self,
              x: Float[Tensor, "B S D"],
              idx: Int[Tensor, "B S"]) -> Float[Tensor, "B S D"]:
    out = x + self.pos_emb[idx]
    return out


class RoPE(nn.Module):
  def __init__(self, dim: int, base: int = 10_000):
    super().__init__()
    self.dim = dim
    self.base = base

    # theta = base^{-i / dim} where i in [0, dim//2)
    # theta = exp(log(base^{-i/ dim})) = exp(-log(base) * (i/dim))
    self.register_buffer(
        "theta",
        torch.exp(-math.log(self.base) * 2 * torch.arange(0, dim//2) / dim)) # shape: (D//2,)


  def forward(self,
              x: Float[Tensor, "B S D"],
              idx: Int[Tensor, "B S"]) -> Float[Tensor, "B S D"]:
    th = einsum(idx, self.theta, "b s, d -> b s d") # [B, S, D//2]
    cos_th = torch.cos(th)
    sin_th = torch.sin(th)
    half_dim = self.dim // 2
    out = torch.concat([cos_th * x[..., :half_dim] - sin_th * x[..., half_dim:],
                        sin_th * x[..., :half_dim] + cos_th * x[..., half_dim:]], dim=-1)
    return out


def create_position_embedding(dim: int, max_seq_len: int, method: str):
  assert method in ["learnable", "absolute", "rope"]
  if method == "learnable":
    return LearnablePositionEmbedding(dim, max_seq_len)
  elif method == "absolute":
    return AbsolutePositionEmbedding(dim, max_seq_len)
  elif method == "rope":
    return RoPE(dim)
  else:
    raise ValueError(f"Unknown supported embedding method: {method}")



In [ ]:
%%ipytest


import matplotlib.pyplot as plt


def test_learnable_position_embedding():
  dim = 64
  max_seq_len = 100
  emb = LearnablePositionEmbedding(dim, max_seq_len)

  batch_size, seq_len = 8, 32
  x = torch.randn((batch_size, seq_len, dim))
  idx = torch.stack([torch.arange(0, seq_len, dtype=torch.long)] * batch_size)
  out = emb(x, idx)
  assert out.shape == x.shape

def test_create_sinusoidal_position_embedding():
  dim = 64
  max_seq_len = 100
  emb = create_sinusoidal_position_embedding(dim, max_seq_len)
  assert emb.shape == (max_seq_len, dim)


  batch_size, seq_len = 32, 64
  idx = torch.randint(low=0, high=max_seq_len, size=(batch_size, seq_len))
  out = emb[idx]
  assert out.shape == (batch_size, seq_len, dim)

  for b_idx in range(batch_size):
    for s_idx in range(seq_len):
      assert torch.all(out[b_idx, s_idx] == emb[idx[b_idx, s_idx]])

  plt.figure()
  plt.imshow(emb.numpy())
  plt.xlabel("feature dimension")
  plt.ylabel("index in sequence")
  plt.title("sinusoidal position embedding")


def test_create_position_embedding():
  dim = 64
  max_seq_len = 100
  emb = create_position_embedding(dim, max_seq_len, "absolute")
  emb = create_position_embedding(dim, max_seq_len, "learnable")
  emb = create_position_embedding(dim, max_seq_len, "rope")


def test_absolute_position_embedding():
  dim = 64
  max_seq_len = 100
  batch_size = 8

  embedder = AbsolutePositionEmbedding(dim, max_seq_len)
  idx = torch.stack([torch.arange(0, max_seq_len, dtype=torch.long)] * batch_size)  # [B, S]
  x = torch.zeros((batch_size, max_seq_len, dim))
  emb = embedder(x, idx)
  plt.figure()
  plt.subplot(121)
  plt.imshow(emb[0, ...].numpy())
  plt.xlabel("feature dimension")
  plt.ylabel("index in sequence")
  plt.title("Absolute PE of a full-length sequence")

  idx_half_seq = torch.stack([torch.arange(0, max_seq_len // 2, dtype=torch.long)] * batch_size)  # [B, S]
  x_half_seq = torch.zeros((batch_size, max_seq_len // 2, dim))
  emb_half_seq = embedder(x_half_seq, idx_half_seq)
  plt.subplot(122)
  plt.xlabel("feature dimension")
  plt.ylabel("index in sequence")
  plt.imshow(emb_half_seq[0, ...].numpy())
  plt.title("Absolute PE of a half-length sequence")


def test_rope():
  dim = 64
  max_seq_len = 100
  batch_size = 8
  rope = RoPE(dim)
  x = torch.randn((batch_size, max_seq_len, dim))
  idx = torch.stack([torch.arange(0, max_seq_len, dtype=torch.long)] * batch_size)  # [B, S]
  out = rope(x, idx)
  assert out.shape == x.shape




### MoE

Implement a simplistic MoE layer



### Transformer!

Define the transformer architecture using the layers defind above.

In [ ]:
from numpy import zeros
from collections import defaultdict

class Transformer(nn.Module):
  def __init__(self,
               num_layers: int,
               dim: int,
               heads: int,
               max_seq_len: int,
               vocab_size: int,
               enable_weight_tying: bool,
               position_embedding_type: str,
               dropout: float,
               num_experts: int = -1,
               num_active_experts: int = -1):
    """

    enable_weight_tying: If true, the output layer will share the same weight as the embedding layer.
    """

    super().__init__() # Initialize the parent class (nn.Module)
    self.num_layers = num_layers
    self.dim = dim
    self.max_seq_len = max_seq_len
    self.vocab_size = vocab_size

    self.num_experts = num_experts

    self.embedder = nn.Embedding(vocab_size, dim)

    logger.info(f"Using {position_embedding_type} positional embedding!")
    self.position_embedding = create_position_embedding(
        dim=dim,
        max_seq_len=max_seq_len,
        method=position_embedding_type
    )

    # It seems `self.layers` was not initialized as a list in the original code.
    # Let's initialize it as a ModuleList to properly register the layers.
    self.layers = nn.ModuleList()
    for i in range(num_layers):
      # For now, using Identity, but this would be where actual transformer blocks go.
      self.layers.append(TransformerLayer(dim=dim,
                                          heads=heads,
                                          max_seq_len=max_seq_len,
                                          use_causal_mask=True,
                                          dropout=dropout,
                                          num_experts=num_experts,
                                          num_active_experts=num_active_experts))

    self.prenorm_output = nn.RMSNorm(dim)
    self.output_layer = nn.Linear(dim, vocab_size, bias=False)

    if enable_weight_tying:
      self.output_layer.weight = self.embedder.weight

    # initialize weights
    self.initialized_modules = defaultdict(int)
    self.apply(self._init_weights)
    logger.info(f"Layers initialized by type:\n{json.dumps(dict(self.initialized_modules), indent=2)}")

  def _init_weights(self, module):
    # TODO (theory): Understand why exactly each layer is initialized this way (and also why not the other way, say Kaiming/Xavier init)
    if isinstance(module, nn.Linear):
      nn.init.normal_(module.weight, mean=0.0, std=0.02)
      if module.bias is not None:
        nn.init.zeros_(module.bias)
      self.initialized_modules["linear"] += 1
    elif isinstance(module, nn.Embedding):
      nn.init.normal_(module.weight, mean=0.0, std=0.02)
      self.initialized_modules["embedding"] += 1
    elif isinstance(module, nn.RMSNorm):
      nn.init.ones_(module.weight)
      assert not hasattr(module, "bias")
      self.initialized_modules["rmsnorm"] += 1
    elif isinstance(module, TransformerLayer):
      module.mlp.apply_init_trick(self.num_layers)
      module.attn.apply_init_trick(self.num_layers)
      self.initialized_modules["transformer_layer"] += 1
      logger.info(f"Applied the sigma=0.02/sqrt(2 * num_layers) trick to initialize linear layers closing residual connections!")

  def forward(self, token_ids: Int[Tensor, "B S"], idx: Int[Tensor, "B S"]):
    """
    """
    x = self.embedder(token_ids)
    x = self.position_embedding(x, idx)

    loads = {}
    for i, layer in enumerate(self.layers):
      x, load = layer(x)
      loads[i] = load

    logits = self.output_layer(self.prenorm_output(x))
    return logits, loads

In [ ]:
%%ipytest

def test_transformer():
  vocab_size = 100
  batch_size, max_seq_len = 8, 32
  dim, heads = 32, 4
  enable_weight_tying = True

  model = Transformer(
      num_layers=8,
      dim=dim,
      heads=heads,
      max_seq_len=max_seq_len,
      vocab_size=vocab_size,
      enable_weight_tying=enable_weight_tying,
      position_embedding_type="rope",
      dropout=0.1)


  device = "cuda:0" if torch.cuda.is_available() else "cpu"
  logger.info(f"{device=}")
  model.to(device)

  # dummy token ids
  token_ids = torch.randint(
      low=0, high=vocab_size,
      size=(batch_size, max_seq_len)).to(device)
  idx = torch.stack([torch.arange(0, max_seq_len, dtype=torch.long)] * batch_size).to(device)
  out, loads = model(token_ids, idx)

  assert out.shape == (batch_size, max_seq_len, vocab_size)
  assert not enable_weight_tying or  model.output_layer.weight is model.embedder.weight

def test_transformer_with_moe():
  vocab_size = 100
  batch_size, max_seq_len = 8, 32
  dim, heads = 32, 4
  enable_weight_tying = True

  model = Transformer(
      num_layers=8,
      dim=dim,
      heads=heads,
      max_seq_len=max_seq_len,
      vocab_size=vocab_size,
      enable_weight_tying=enable_weight_tying,
      position_embedding_type="rope",
      dropout=0.1,
      num_experts=4,
      num_active_experts=2)


  device = "cuda:0" if torch.cuda.is_available() else "cpu"
  logger.info(f"{device=}")
  model.to(device)

  # dummy token ids
  token_ids = torch.randint(
      low=0, high=vocab_size,
      size=(batch_size, max_seq_len)).to(device)
  idx = torch.stack([torch.arange(0, max_seq_len, dtype=torch.long)] * batch_size).to(device)
  out, loads = model(token_ids, idx)

  assert out.shape == (batch_size, max_seq_len, vocab_size)
  assert not enable_weight_tying or  model.output_layer.weight is model.embedder.weight

### Validation function



In [ ]:
from copy import deepcopy

def validate(model,
             text: str,
             tokenizer: CharacterTokenizer,
             device: str,
             temperature: float = 0.0,
             horizon: int = 32):
  model.eval()

  token_ids =  tokenizer.tokenize(text)
  pred_text = deepcopy(text)
  for _ in range(horizon):
    tensor_token_ids = torch.tensor(token_ids, dtype=torch.long, device=device).unsqueeze(0)  # [1, S]
    tensor_idx = torch.arange(0, len(token_ids), dtype=torch.long, device=device).unsqueeze(0)  # [1, S]
    with torch.no_grad():
      logits, _ = model(token_ids=tensor_token_ids, idx=tensor_idx)  # [B S C]

      last_token_logits = logits[:, -1, :]  # [B C]
    if temperature > 0.0:
      # logits shape: [B, S, C]
      # Reshape to 2D (B*S, C) for torch.multinomial
      last_token_probs = nn.functional.softmax(last_token_logits / temperature, dim=-1) # [B, C]
      last_token_idx = torch.multinomial(last_token_probs, num_samples=1) # [B, 1]
    else:
      last_token_idx = logits[:, -1, :].max(dim=-1)[1]  # [B, 1]

    token_ids.append(last_token_idx.item())

    new_char = tokenizer.detokenize([last_token_idx.item()])
    pred_text += new_char

  return pred_text

In [ ]:
%%ipytest

def test_validate():
  # load vocabulary
  tokenizer = CharacterTokenizer(vocab_path=workspace / "vocab.json")


  batch_size, max_seq_len = 8, 128
  dim, heads = 32, 4
  enable_weight_tying = False

  model = Transformer(
      num_layers=8,
      dim=64,
      heads=8,
      max_seq_len=128,
      vocab_size=tokenizer.vocab_size,
      enable_weight_tying=False,
      position_embedding_type="absolute",
      dropout=0.1)
  model.eval()


  validate(model,
           text="Hello world",
           tokenizer=tokenizer,
           device="cpu",
           temperature=0.0,
           horizon=32)



### Learning rate scheduler

It's standard to use a linear warmup and then cosine decay for learning rate scheduling.


In [ ]:
def get_lr_lambda(current_step: int, warmup_steps: int, total_steps: int):
    warmup_steps = warmup_steps
    if current_step < warmup_steps:
        return float(current_step) / float(max(1, warmup_steps))

    # Cosine decay
    progress = float(current_step - warmup_steps) / float(max(1, total_steps - warmup_steps))
    progress = min(1.0, max(0.0, progress))
    cosine_decay = 0.5 * (1.0 + math.cos(math.pi * progress))

    # Decay to 10% of the initial LR
    min_lr_ratio = 0.1
    return min_lr_ratio + (1.0 - min_lr_ratio) * cosine_decay

lrs = [get_lr_lambda(step, warmup_steps=500, total_steps=10000) for step in range(10000)]
plt.plot(range(10_000), lrs)
plt.xlabel("training step")
plt.ylabel("learning rate multiplier")
plt.grid("on")
plt.title("learning rate schedule: linear warmup + cosine decay")
plt.show()



## Training loop!

In [ ]:
%env LOGURU_LEVEL=DEBUG

TRAIN: bool = True

########################################
# BEGIN: TOP LEVEL CONFIG
########################################

# ckpt_dir: Path = project_dir / "tiny-shakespeare-ckpts-gpu-feb18-1443"
# restore_ckpt_from: Path | None = ckpt_dir /  "checkpoint-global_step00056000-datetime20260218-1716.pt"

# ckpt_dir: Path = workspace / "checkpoints"/ "debug-tiny-shapespeare-feb19-pytorch-sdpa"
# restore_ckpt_from =  ckpt_dir / "checkpoint-global_step00029500-datetime20260220-0818.pt"

# ckpt_dir: Path = workspace / "checkpoints"/ "debug-tiny-shapespeare-feb20-1101"
# restore_ckpt_from = ckpt_dir / "checkpoint-global_step=00017000.pt"

# ckpt_dir: Path = workspace / "checkpoints"/ "debug-tiny-shapespeare-feb20-1306"
# restore_ckpt_from = None

# Below is a good model and checkpoint!
# ckpt_dir: Path = workspace / "checkpoints"/ "debug-tiny-shapespeare-feb20-1505"
# restore_ckpt_from = ckpt_dir / "checkpoint-global_step=00017000.pt"

# ckpt_dir: Path = workspace / "checkpoints"/ "debug-tiny-shapespeare-feb23-2304"
# restore_ckpt_from = ckpt_dir / "checkpoint-global_step=00006000.pt"

# ckpt_dir: Path = workspace / "checkpoints"/ "debug-rope-tiny-shapespeare-feb23-2350"
# restore_ckpt_from = ckpt_dir / "checkpoint-global_step=00003000.pt"

ckpt_dir: Path = workspace / "checkpoints"/ "debug-moe-tiny-shapespeare-feb26-0857"
restore_ckpt_from = ckpt_dir / "checkpoint-global_step=00007000.pt"

# ckpt_dir: Path = workspace / "checkpoints"/ "debug-moe-tiny-shapespeare-feb26-0857-no-load-balance"
# restore_ckpt_from = ckpt_dir / "checkpoint-global_step=00002000.pt"

if not ckpt_dir.exists():
  ckpt_dir.mkdir(parents=True, exist_ok=True)
  logger.info(f"checkpoint directory created @ {ckpt_dir}")

# restore checkpoint if needed
restored_ckpt = None
if restore_ckpt_from is not None:
  restored_ckpt = torch.load(restore_ckpt_from)
  logger.info(f"Restored checkpoint from {restore_ckpt_from}")

train_data = text # train on the full dataset
# train_data = "\n".join([text[:128] * 10_000]) # overfit for debugging

########################################
# END: TOP LEVEL CONFIG
########################################

import os
import sys
from dataclasses import dataclass, asdict
import datetime
from zoneinfo import ZoneInfo
from pprint import pprint
import math

from torch.optim import AdamW

logger.remove()
logger.add(sys.stderr, level=os.environ["LOGURU_LEVEL"])

device = "cuda:0" if torch.cuda.is_available() else "cpu"
logger.info(f"{device=}")

def get_datetime_string():
  # Get the current datetime object
  now = datetime.datetime.now(tz=ZoneInfo("America/Los_Angeles"))

  # Format the datetime object as YYYYMMDD-HHMM
  formatted_time = now.strftime("%Y%m%d-%H%M")
  return formatted_time

# define model config
@dataclass
class ModelConfig:
  num_layers: int = 6
  dim: int = 256
  heads: int = 4
  max_seq_len: int = 256
  enable_weight_tying: bool = False
  position_embedding_type: str = "rope"
  dropout: float = 0.2
  # MoE setting
  num_experts: int = 2
  num_active_experts: int = 1

# define trainer config
@dataclass
class TrainerConfig:
  seed: int = 42
  num_epochs: int = 1
  batch_size: int = 64
  lr: float = 1e-3
  # The value of beta1, beta2, eps, and weight_decay are set to Pytorch default.
  # But we can tune them through this config if needed.
  beta1: float = 0.9
  beta2: float = 0.95
  eps: float = 1e-8
  weight_decay: float = 0.01
  # checkpointing
  ckpt_every_n_steps: int = 1000  # -> 1000 for non-debug run
  # other logging
  log_loss_every_n_steps: int = 50
  # scheduler
  warmup_steps: int = 1000
  # load balancing weight
  load_balancing_weight: float = 0.0

if restored_ckpt:
  model_config = ModelConfig(**restored_ckpt["model_config"])
  trainer_config = TrainerConfig(**restored_ckpt["trainer_config"])
  logger.info("Configuration restored from checkpoint!")
else:
  model_config = ModelConfig()
  trainer_config = TrainerConfig()
  logger.info("Using default configuration!!")

pprint(model_config, indent=2)
pprint(trainer_config, indent=2)


# create character tokenizer
collate_fn, tokenizer = create_collate_fn()

# create model
model = Transformer(
    num_layers=model_config.num_layers,
    dim=model_config.dim,
    heads=model_config.heads,
    max_seq_len=model_config.max_seq_len,
    vocab_size=tokenizer.vocab_size,
    enable_weight_tying=model_config.enable_weight_tying,
    position_embedding_type=model_config.position_embedding_type,
    dropout=model_config.dropout,
    num_experts=model_config.num_experts,
    num_active_experts=model_config.num_active_experts
)

model.to(device)
logger.info(f"Model moved to {device}")

if restored_ckpt:
  model.load_state_dict(restored_ckpt["model"])
  logger.info("Model state restored!")

if TRAIN:
  # training mode

  model.train()

  # create optimizer
  optimizer = AdamW(model.parameters(),
                    lr=trainer_config.lr,
                    betas=(trainer_config.beta1, trainer_config.beta2),
                    eps=trainer_config.eps,
                    weight_decay=trainer_config.weight_decay)

  if restored_ckpt:
    optimizer.load_state_dict(restored_ckpt["optimizer"])
    logger.info("Optimizer state restored!")

  # create dataloader
  if restored_ckpt is None:
    dataset = TinyTextDataset(
        train_data,
        seq_len=model.max_seq_len + 1,  # +1 so that we can use seq[:-1] as input and seq[1:] as target
        num_epochs=trainer_config.num_epochs,
        rng_seed=trainer_config.seed)
  else:
    dataset_state = restored_ckpt.get("dataset", None)
    assert dataset_state is not None
    dataset = TinyTextDataset.create_from_state(dataset_state)

  dataloader = DataLoader(
      dataset,
      batch_size=trainer_config.batch_size,
      shuffle=False, # shuffle=False for reproducible demonstration
      collate_fn=collate_fn
  )

  # restore the global step if available
  if restored_ckpt is None:
    global_step = 0
  else:
    global_step = restored_ckpt.get("global_step", 0)

  # Init Scheduler
  total_steps = len(dataloader)
  logger.info(f"{total_steps=}")

  scheduler = torch.optim.lr_scheduler.LambdaLR(
      optimizer,
      lr_lambda=lambda step: get_lr_lambda(step, warmup_steps=trainer_config.warmup_steps, total_steps=total_steps),
      last_epoch=global_step - 1)

  if restored_ckpt:
      scheduler.load_state_dict(restored_ckpt["scheduler"])
      logger.info("Scheduler state restored!")


  # pbar stands for progress bar
  pbar = tqdm.tqdm(dataloader, unit="batch", initial=global_step)

  for batch in pbar:
    token_ids = batch["token_ids"].to(device) # [B, S + 1]
    position_idx = batch["position_idx"].to(device) # [B, S + 1] -- index of each token's position in the sequence

    logits, loads = model(token_ids=token_ids[:, :-1], idx=position_idx[:, :-1]) # [B, S, C] where C stands for classes

    # compute the next token prediction loss (cross entropy)
    loss = nn.functional.cross_entropy(
        input=rearrange(logits,
                        "b s c -> (b s) c"),
        target=rearrange(token_ids[:, 1:],  # shift to next token
                        "b s -> (b s)"),
        reduction="mean"
    )

    # compute the load balancing loss for MoE (if available)
    if model.num_experts > 0:
      loss_load_balancing = torch.mean(
          torch.tensor([model.num_experts * torch.dot(load.P, load.f) for _, load in loads.items() if load is not None])
      )
    else:
      loss_load_balancing = torch.tensor(0.0, device=device, dtype=loss.dtype)

    loss += trainer_config.load_balancing_weight * loss_load_balancing

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    scheduler.step()

    global_step += 1

    lr = scheduler.get_last_lr()[0]
    pbar.set_postfix({"loss": f"{loss.item():0.4f}", "loss_lb": f"{loss_load_balancing.item():0.4f}", "lr": f"{lr:.2e}"})

    current_datetime = get_datetime_string()

    if global_step % trainer_config.log_loss_every_n_steps == 0:
      with open(ckpt_dir / "loss.txt", "a") as f:
        f.write(f"{global_step=:08d}, {current_datetime=}, loss={loss.item():0.4f}, lb_loss={loss_load_balancing.item():0.4f}, lr={lr:.2e}\n")
        f.flush()

      # pprint(loads)

    if global_step % trainer_config.ckpt_every_n_steps == 0:
      name = f"{global_step=:08d}"

      torch.save({"model": model.state_dict(),
                  "optimizer": optimizer.state_dict(),
                  "scheduler": scheduler.state_dict(),
                  "global_step": global_step,
                  "loss": loss.item(),
                  "loss_load_balancing": loss_load_balancing.item(),
                  "dataset": dataset.serialize_state(),
                  "model_config": asdict(model_config),
                  "trainer_config": asdict(trainer_config)
                  },
                ckpt_dir / f"checkpoint-{name}.pt")

      # validate model
      with open(ckpt_dir / "validation.txt", "a") as f:
        val_text = "First Citizen:\nBefore we proceed any further, hear me speak."
        pred_text = validate(model,
                     text=val_text,
                     tokenizer=tokenizer,
                     device=device,
                     temperature=0.0, horizon=64)
        f.write(f"{global_step=:08d}, {current_datetime=}\n")
        f.write(f"{repr(val_text)}\n{repr(pred_text)}\n")
        model.train() # restore to train mode
else:
  # inference mode
  model.eval()
  logger.info("Model is ready for inference!")

## Inference

Test a checkpoint

In [ ]:
from copy import deepcopy


# input_text = """ROMEO:
# Is the day so young?

# BENVOLIO:
# """
# print(repr(input_text))
offset = 0
input_text = text[offset:offset + 32]

pred_text = validate(model,
                     text=input_text,
                     char_to_idx_vocab=vocab,
                     idx_to_char_vocab=idx_to_char_vocab,
                     device=device,
                     temperature=0.0, horizon=64)
print(f"{repr(input_text)}\n{repr(pred_text)}\n")

